In [1]:
!apt-get -qq -y install fonts-nanum > /dev/null

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost lightgbm catboost

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

PROCESSED_DIR = Path("/content/drive/MyDrive/baram2026/processed")
OUTPUT_DIR = Path("/content/drive/MyDrive/baram2026/submissions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(PROCESSED_DIR / "train_features_v1.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test_features_v1.parquet")
with open(PROCESSED_DIR / "feature_cols_v1.json") as f:
    feature_cols = json.load(f)

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}

print("train_df:", train_df.shape, " test_df:", test_df.shape, " feature 수:", len(feature_cols))

train_df: (26304, 154)  test_df: (8760, 152)  feature 수: 150


In [4]:
def metric(answer_df, pred_df):
    group_nmae, group_ficr = [], []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]
        valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    one_minus_nmae = 1 - np.mean(group_nmae)
    ficr = np.mean(group_ficr)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr


val_months = [1, 4, 7, 10]
is_val = train_df["forecast_kst_dtm"].dt.month.isin(val_months)
is_tr = ~is_val

print("학습 구간:", is_tr.sum(), " / 검증 구간:", is_val.sum())
print("검증 구간 연도별 분포:")
print(train_df.loc[is_val, "forecast_kst_dtm"].dt.year.value_counts().sort_index())

학습 구간: 17448  / 검증 구간: 8856
검증 구간 연도별 분포:
forecast_kst_dtm
2022    2951
2023    2952
2024    2952
2025       1
Name: count, dtype: int64


In [5]:
imputer = SimpleImputer(strategy="median")
X_all_imp = pd.DataFrame(
    imputer.fit_transform(train_df[feature_cols]), columns=feature_cols, index=train_df.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(test_df[feature_cols]), columns=feature_cols, index=test_df.index
)

In [6]:
MODEL_NAMES = ["RandomForest", "XGBoost", "LightGBM", "CatBoost", "HistGradientBoosting"]
MODEL_USES_RAW_NA = {"RandomForest": False, "XGBoost": True, "LightGBM": True, "CatBoost": True, "HistGradientBoosting": True}

def build_model(name):
    if name == "RandomForest":
        return RandomForestRegressor(random_state=42, n_jobs=-1)
    if name == "XGBoost":
        return XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
    if name == "LightGBM":
        return LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    if name == "CatBoost":
        return CatBoostRegressor(random_seed=42, verbose=False)
    if name == "HistGradientBoosting":
        return HistGradientBoostingRegressor(random_state=42)

In [7]:
import time

results = []

for model_name in MODEL_NAMES:
    val_pred_df = pd.DataFrame(index=train_df.index[is_val])
    t0 = time.time()

    for target in TARGET_COLS:
        train_mask = train_df[target].notna() & is_tr

        if MODEL_USES_RAW_NA[model_name]:
            X_train_fold = train_df.loc[train_mask, feature_cols]
            X_val_fold = train_df.loc[is_val, feature_cols]
        else:
            X_train_fold = X_all_imp.loc[train_mask]
            X_val_fold = X_all_imp.loc[is_val]

        y_train_fold = train_df.loc[train_mask, target]

        model = build_model(model_name)
        model.fit(X_train_fold, y_train_fold)

        pred = model.predict(X_val_fold)
        val_pred_df[target] = np.clip(pred, 0, CAPACITY_KWH[target])

    elapsed = time.time() - t0
    answer_val = train_df.loc[is_val, TARGET_COLS].reset_index(drop=True)
    pred_val = val_pred_df.reset_index(drop=True)
    score, one_minus_nmae, ficr = metric(answer_val, pred_val)

    results.append({"model": model_name, "score": score, "1-NMAE": one_minus_nmae, "FICR": ficr, "elapsed_sec": round(elapsed, 1)})
    print(f"[{model_name}] score={score:.5f} 1-NMAE={one_minus_nmae:.5f} FICR={ficr:.5f} ({elapsed:.1f}s)")

results_df = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)
results_df

[RandomForest] score=0.56841 1-NMAE=0.85455 FICR=0.28226 (1079.3s)
[XGBoost] score=0.56687 1-NMAE=0.84680 FICR=0.28694 (24.2s)
[LightGBM] score=0.57583 1-NMAE=0.85443 FICR=0.29724 (7.9s)
[CatBoost] score=0.57199 1-NMAE=0.85408 FICR=0.28990 (146.6s)
[HistGradientBoosting] score=0.57543 1-NMAE=0.85496 FICR=0.29591 (11.7s)


,model,score,1-NMAE,FICR,elapsed_sec
0,LightGBM,0.575834,0.854426,0.297242,7.9
1,HistGradientBoosting,0.575434,0.854961,0.295908,11.7
2,CatBoost,0.571993,0.854085,0.289901,146.6
3,RandomForest,0.568408,0.854551,0.282264,1079.3
4,XGBoost,0.566871,0.846800,0.286942,24.2


In [8]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

for model_name in MODEL_NAMES:
    predictions = pd.DataFrame(index=test_df.index)

    for target in TARGET_COLS:
        train_mask = train_df[target].notna()

        if MODEL_USES_RAW_NA[model_name]:
            X_train_full = train_df.loc[train_mask, feature_cols]
            X_test_full = test_df[feature_cols]
        else:
            X_train_full = X_all_imp.loc[train_mask]
            X_test_full = X_test_imp

        y_train_full = train_df.loc[train_mask, target]

        model = build_model(model_name)
        model.fit(X_train_full, y_train_full)

        pred = model.predict(X_test_full)
        predictions[target] = np.clip(pred, 0, CAPACITY_KWH[target])

    submission = test_df[["forecast_id", "forecast_kst_dtm"]].copy()
    for target in TARGET_COLS:
        submission[target] = predictions[target]
    submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

    output_path = OUTPUT_DIR / f"submit_{model_name}_{timestamp}.csv"
    submission.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"[{model_name}] 저장 완료: {output_path}  (shape={submission.shape})")

[RandomForest] 저장 완료: /content/drive/MyDrive/baram2026/submissions/submit_RandomForest_20260717_1318.csv  (shape=(8760, 5))
[XGBoost] 저장 완료: /content/drive/MyDrive/baram2026/submissions/submit_XGBoost_20260717_1318.csv  (shape=(8760, 5))
[LightGBM] 저장 완료: /content/drive/MyDrive/baram2026/submissions/submit_LightGBM_20260717_1318.csv  (shape=(8760, 5))
[CatBoost] 저장 완료: /content/drive/MyDrive/baram2026/submissions/submit_CatBoost_20260717_1318.csv  (shape=(8760, 5))
[HistGradientBoosting] 저장 완료: /content/drive/MyDrive/baram2026/submissions/submit_HistGradientBoosting_20260717_1318.csv  (shape=(8760, 5))
